# Applied Machine Learning, Notebook 6
## Credit risk in Taiwan: beyond logistic regression

**Course:** *Applied Machine Learning* · **Prepared by:** David Díaz, with assistance from Claude (Anthropic) · **Environment:** Google Colab

### Where we come from and where we are going

In the previous notebook we learned to predict a yes or a no with logistic regression, and above all we learned
to **evaluate** a classifier: confusion matrix, recall and precision, the threshold as a decision, the ROC curve and
the AUC, the cost-optimal threshold, and how much all of that moves if the sample changes. All with 1,000 German
customers, where a straight line was enough and no embellishment improved it.

Today we change dataset: **30,000 credit card customers of a Taiwanese bank** (Yeh and Lien, 2009), with
six months of payment and delay history. And we change the question: are there models that beat the logistic, and
why? We are going to try three new things, each explained from scratch: the logistic with **polynomial
terms** (where regularization, which was useless in the previous notebook, now does matter),
**decision trees** (CART, drawn box by box, and C5.0 with the `c50py` library), and the two tree ensembles that
dominate prediction with tabular data, **random forest** and **gradient boosting**.

The course rule does not change: each model is measured in testing, with its interval, and before declaring a winner
we check that the difference is larger than the noise. And when a model wins, we are going to see with our
own eyes **what it found** that the logistic could not see.

### What you will learn today

1. When regularization does improve testing: degree-2 polynomials with and without a brake.
2. What a decision tree is, why it draws steps and why it overfits.
3. What an ensemble is: random forest (many independent trees averaged) and gradient boosting (small trees that correct each other in sequence).
4. How to compare models without fooling yourself: cross-validation, bootstrap intervals and the paired difference.
5. How to look inside the winner: the nonlinear shape it found, and the cost-based threshold for using it.


In [ ]:

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings, time

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report, recall_score, precision_score

warnings.filterwarnings("ignore")
pd.set_option("display.width", 130)


## 1. The data: credit cards in Taiwan

Thirty thousand credit card customers in 2005, with their credit limit, demographic data and **six months
of history**: how late they were each month, how much they were billed and how much they paid. The target is whether they defaulted
the following month ($y = 1$): it happened to 22%. The columns come as `x1`, `x2`; we rename them so we can
read them. If you have the course file (`taiwan_credit.csv`), paste its link into `URL_TAIWAN`.

The variable that is going to matter most is `atraso_sep`, the payment status in the most recent month: negative values
or zero mean they paid on time (with nuances that do not matter today), 1 means one month late, 2 two
months, and so on up to 8.


In [ ]:

URL_TAIWAN = ""    # <-- paste here the link to the course CSV (taiwan_credit.csv) if you have it

def normalizar_link(url):
    if "dropbox.com" in url:
        return url.replace("dl=0", "dl=1").replace("www.dropbox.com", "dl.dropboxusercontent.com")
    if "drive.google.com" in url and "/d/" in url:
        return "https://drive.google.com/uc?export=download&id=" + url.split("/d/")[1].split("/")[0]
    return url

if URL_TAIWAN:
    tw = pd.read_csv(normalizar_link(URL_TAIWAN))
    X = tw.drop(columns=tw.columns[-1]).apply(pd.to_numeric)
    y = tw[tw.columns[-1]].astype(int).values
else:
    tw = fetch_openml("default-of-credit-card-clients", version=1, as_frame=True)
    X = tw.data.apply(pd.to_numeric)
    y = tw.target.astype(int).values

meses = ["sep", "ago", "jul", "jun", "may", "abr"]
X.columns = (["limite_credito", "sexo", "educacion", "estado_civil", "edad"] + ["atraso_" + m for m in meses]
             + ["factura_" + m for m in meses] + ["pago_" + m for m in meses])
print("Customers:", len(X), "  Columns:", X.shape[1], "  Proportion that defaulted:", round(y.mean(), 3))
X[["limite_credito", "edad", "atraso_sep", "atraso_ago", "factura_sep", "pago_sep"]].describe().round(0).T[["mean", "min", "50%", "max"]]


### The same ritual: split, and scale using training only

70% for training (21,000 customers) and 30% for measuring (9,000), stratified. We scale with the mean and the
standard deviation of training. Trees do not need scaling (they decide with thresholds, and a threshold does not change if the
variable is multiplied by a constant), but it does not hurt them, so to keep things simple all the models
receive the same scaled columns.


In [ ]:

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
esc = StandardScaler().fit(X_tr)
A_tr, A_te = esc.transform(X_tr), esc.transform(X_te)
print("training:", len(y_tr), "customers,", int(y_tr.sum()), "defaulted  |  testing:", len(y_te), "customers,", int(y_te.sum()), "defaulted")


## 2. The evaluation kit, summarized

From the previous notebook we bring three things, packaged in a function so we can use them with each model:

- The **AUC** in training and in testing: how the model ranks customers by risk, and the gap between the
  two, which is the thermometer of how much it memorized.
- The **recall and precision of class 1** (does not pay) with threshold 0.5, to connect with the confusion matrix.
- The **capture at 20%**: if the bank rejects the 20% the model considers riskiest, what percentage of
  those who actually defaulted was rejected. It is the AUC translated into a decision a manager understands;
  at random it would be 20%.

And a fourth one we are going to need at the end: the **bootstrap interval** of the testing AUC (resample the
9,000 customers with replacement, recompute, repeat). With 9,000 customers the interval should be much
narrower than the German Credit one, which went from 0.76 to 0.87 with 300.


In [ ]:

def captura(y_real, riesgo, fraccion=0.20):
    cuantos = int(len(y_real) * fraccion)
    orden = np.argsort(-riesgo)
    return y_real[orden[:cuantos]].sum() / y_real.sum()

rng_bs = np.random.default_rng(0)
IDX_BS = [rng_bs.integers(0, len(y_te), len(y_te)) for _ in range(500)]   # the same resamples for every model

def ic_auc(p):
    v = np.array([roc_auc_score(y_te[i], p[i]) for i in IDX_BS])
    return np.percentile(v, 2.5), np.percentile(v, 97.5)

RESULTADOS = {}
def evaluar(nombre, p_tr, p_te, segundos=None):
    pred = (p_te > 0.5).astype(int)
    lo, hi = ic_auc(p_te)
    RESULTADOS[nombre] = dict(auc_train=roc_auc_score(y_tr, p_tr), auc_test=roc_auc_score(y_te, p_te), ic_lo=lo, ic_hi=hi,
                              recall_1=recall_score(y_te, pred), precision_1=precision_score(y_te, pred), captura_20=captura(y_te, p_te), p_te=p_te, seg=segundos)
    r = RESULTADOS[nombre]
    print(f"{nombre:38s} AUC train {r['auc_train']:.3f}  test {r['auc_test']:.3f}  [{lo:.3f}, {hi:.3f}]   gap {r['auc_train']-r['auc_test']:+.3f}   "
          f"recall {r['recall_1']:.3f}  precision {r['precision_1']:.3f}  capture20 {r['captura_20']:.3f}" + (f"   ({segundos:.0f} s)" if segundos else ""))


## 3. The usual logistic, and the logistic with polynomials

### Starting point

The same logistic regression as in the previous notebook, unregularized. It is the model against which everything
else is measured.


In [ ]:

t0 = time.time()
logit = LogisticRegression(penalty=None, max_iter=5000).fit(A_tr, y_tr)
evaluar("linear logistic", logit.predict_proba(A_tr)[:, 1], logit.predict_proba(A_te)[:, 1], time.time() - t0)


### Degree-2 polynomials: where regularization does matter

The logistic assumes that each variable always pushes the risk in the same direction and at the same pace, and that
the variables do not interact. One way to relax that without changing model is to add columns: the square of
each variable and the product of each pair. With 23 variables that gives 299 columns ($23$ originals, $23$ squares and
$23 \cdot 22 / 2 = 253$ products). The logistic is still a straight line, but now in 275 dimensions, and that
line can draw curves and crossings in the 23 originals.

The price is that 299 weights are many more degrees of freedom, and with them the model can use that freedom to
fit the noise. We look at it with two versions: degree 2 unregularized, and degree 2 with the $\lambda$
chosen by 5-fold cross-validation on training (`LogisticRegressionCV` tries several values of
$C = 1/\lambda$ and keeps the best one without looking at testing). And then we repeat the experiment with only 1,000
training customers, because the underlying question is how many weights the available information can pay for.


In [ ]:

poly = PolynomialFeatures(degree=2, include_bias=False)
P_tr = poly.fit_transform(A_tr); P_te = poly.transform(A_te)
esc2 = StandardScaler().fit(P_tr)
P_tr, P_te = esc2.transform(P_tr), esc2.transform(P_te)
print("Degree-2 columns:", P_tr.shape[1])

t0 = time.time()
poli_libre = LogisticRegression(penalty=None, max_iter=3000).fit(P_tr, y_tr)
evaluar("logistic degree 2, unregularized", poli_libre.predict_proba(P_tr)[:, 1], poli_libre.predict_proba(P_te)[:, 1], time.time() - t0)

t0 = time.time()
poli_cv = LogisticRegressionCV(Cs=8, cv=5, max_iter=3000, n_jobs=-1).fit(P_tr, y_tr)
evaluar("logistic degree 2, L2 by cross-validation", poli_cv.predict_proba(P_tr)[:, 1], poli_cv.predict_proba(P_te)[:, 1], time.time() - t0)
print("C chosen by cross-validation:", poli_cv.C_[0], " (lambda =", round(1 / poli_cv.C_[0], 4), ")")


Two readings. First, degree 2 beats the straight line by 0.035 of AUC (0.750 against 0.715), with an interval that
does not touch zero: on this dataset **there is something nonlinear to find**. Second, with 21,000 customers not even 299
columns overfit much: the unregularized gap is 0.031 and regularization reduces it to 0.023 while gaining
barely 0.002 of AUC in testing. It is German Credit again, with more data. So, when does
regularization matter? When the weights are many **relative to the rows**. We check it with the same degree-2
model trained with only 1,000 customers:


In [ ]:

P_chico, y_chico = P_tr[:1000], y_tr[:1000]
libre_chico = LogisticRegression(penalty=None, max_iter=5000).fit(P_chico, y_chico)
cv_chico = LogisticRegressionCV(Cs=8, cv=5, max_iter=5000, n_jobs=-1).fit(P_chico, y_chico)
for nombre, m in [("degree 2 with 1,000 customers, unregularized", libre_chico), ("degree 2 with 1,000 customers, L2 by cross-validation", cv_chico)]:
    print(f"{nombre:55s} AUC train {roc_auc_score(y_chico, m.predict_proba(P_chico)[:, 1]):.3f}   test {roc_auc_score(y_te, m.predict_proba(P_te)[:, 1]):.3f}")
print("lambda chosen:", round(1 / cv_chico.C_[0], 3))


There is the difference that did not show up in German Credit: with 1,000 rows and 299 weights, the model without a brake gets
almost all of training right and collapses in testing; the regularized one sacrifices training and generalizes much better.
Regularization is insurance against having more weights than information, and its value depends on that ratio, not on the
model. With abundant data the insurance is barely used; with scarce data it is the only thing that saves the model.

Let's go back to the 21,000 customers. Degree 2 improved on the straight line, so there is a nonlinear shape. The question is
what shape it has, and that is what trees are for.


## 4. Decision trees, from scratch

A **decision tree** is a sequence of yes-or-no questions about the variables. It starts with all the
customers at the root and looks for the question that best separates those who pay from those who do not (for example, "September
delay greater than or equal to 2?"); it splits the customers into two groups according to the answer, and in each group
it looks again for the best question, until it reaches small or sufficiently pure groups. Each final group
(**leaf**) predicts the fraction of customers within it who did not pay.

How it chooses the question: it tries every variable and every possible cut, and keeps the one that most
reduces the **impurity** of the two resulting groups (the usual measure is the Gini index, $1 - \sum_k p_k^2$ with $p_k$ the fraction of each class
in the group, which with two classes reduces to $2p(1-p)$: it is 0 if the group is all one class and 0.5 if it is half and half). There are no weights, no line, no need to
scale: the tree only asks and splits.

Two consequences you need to be clear about. A tree draws **steps**, never ramps: within each leaf the
risk is constant. That is an advantage when reality has jumps (a threshold beyond which everything
changes) and a disadvantage when reality is smooth. And a **deep tree memorizes**: if we let it grow
until each leaf has two customers, it gets 100% of training right and little of testing. The maximum depth
is the brake, and it is chosen the same way as $\lambda$: with cross-validation.


In [ ]:

print("Depth, AUC by cross-validation in training (5 folds):")
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
for prof in [2, 4, 6, 8, 12, None]:
    a = cross_val_score(DecisionTreeClassifier(max_depth=prof, min_samples_leaf=20, random_state=0), A_tr, y_tr, cv=cv5, scoring="roc_auc", n_jobs=-1)
    print(f"   depth {str(prof):5s}  AUC {a.mean():.3f} +/- {a.std():.3f}")


In [ ]:

arbol = DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, random_state=0).fit(A_tr, y_tr)
evaluar("decision tree, depth 4", arbol.predict_proba(A_tr)[:, 1], arbol.predict_proba(A_te)[:, 1])
arbol_hondo = DecisionTreeClassifier(max_depth=None, random_state=0).fit(A_tr, y_tr)
evaluar("decision tree, no limit", arbol_hondo.predict_proba(A_tr)[:, 1], arbol_hondo.predict_proba(A_te)[:, 1])

# the depth-2 tree, trained on the unscaled columns so the cuts can be read
arbol2 = DecisionTreeClassifier(max_depth=2, min_samples_leaf=20, random_state=0).fit(X_tr, y_tr)
texto = export_text(arbol2, feature_names=list(X.columns), decimals=2, show_weights=True)
print("Depth-2 tree (cuts in original units):")
print(texto)


In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

# The same depth-4 tree, trained on the unscaled columns so the cuts can be read in original units.
# A tree does not change if the columns are scaled (it only asks "greater than a threshold?"), so the AUC is the same.
arbol4 = DecisionTreeClassifier(max_depth=4, min_samples_leaf=20, random_state=0).fit(X_tr, y_tr)
print("Testing AUC, depth-4 tree on raw columns:", round(roc_auc_score(y_te, arbol4.predict_proba(X_te)[:, 1]), 3), " (scaled:", round(RESULTADOS["decision tree, depth 4"]["auc_test"], 3), ")")

fig, ax = plt.subplots(figsize=(12, 5))
plot_tree(arbol2, feature_names=list(X.columns), class_names=["pays", "does not pay"], filled=True, proportion=True, impurity=False, rounded=True, fontsize=10, ax=ax)
ax.set_title("Depth-2 tree: in each box, the question, the fraction of customers that reaches it and the fraction [pays, does not pay]")
plt.show()

fig, ax = plt.subplots(figsize=(30, 11))
plot_tree(arbol4, feature_names=list(X.columns), class_names=["pays", "does not pay"], filled=True, proportion=True, impurity=False, rounded=True, fontsize=7, ax=ax)
ax.set_title("Depth-4 tree, minimum 20 customers per leaf (16 leaves at most)")
plt.show()

How to read the boxes. Each box is a question; the customers who answer "yes" (the condition holds) go to the
left and those who answer "no" go to the right. `samples` is the fraction of the 21,000 training customers that reaches
that box, and `value` is the fraction that pays and the fraction that does not pay within it; the color goes from orange (almost
everyone pays) to blue (most do not pay), and the more intense, the purer the box. The leaves at the bottom are the steps of the
model: each testing customer falls into exactly one leaf and receives as probability the "does not pay" fraction of that leaf.

Notice two things. The root asks about `atraso_sep` with the cut at 1.5 (that is, two months or more of delay), and
the right subtree, with 10% of the customers, concentrates those who do not pay (70% within it, against 17% on the
left). And the first questions on both sides are about the delays of the previous months; the credit limit, age and
the bill amounts only appear at the last level and for small groups. The payment history is what separates, just as
the variable importance and the boosting will say later on.


The depth-2 tree already tells the story: the first question is about the September delay, and the
cut falls between 1 and 2 months late. A single tree, even well pruned, does not beat the degree-2 logistic,
and without a limit it memorizes (look at the gap). What follows is how to turn many mediocre trees into a good
model.


### C5.0: the other classic tree, with `c50py`

The scikit-learn tree is CART (Breiman, Friedman, Olshen and Stone, 1984): binary splits, Gini impurity, and the
depth or the minimum per leaf as the brake. There is another family just as classic, Quinlan's: ID3, C4.5 and its
commercial version **C5.0**, which is what `C50` in R and many rule engines in banks use. It differs from CART in three
things worth knowing:

1. It chooses the question with the **gain ratio** (information gain normalized by the entropy of the
   partition) instead of Gini, which penalizes variables with many distinct values.
2. It treats **categorical variables as categorical**: a question can group several categories in one branch
   ("educacion in {1, 2}"), without converting them to 0/1 columns. Here `sexo`, `educacion` and `estado_civil` come coded as integers and CART does not
   care; C5.0 is told which ones are categories.
3. **Pessimistic pruning**: it lets the tree grow and then trims it from the leaves upward, comparing the error of
   each subtree with an upper bound of its error (the confidence factor `cf`, 0.25 by default). Then a global pruning
   simplifies the whole tree as long as the training errors do not rise by more than one standard error. There is
   no depth to choose; the tree ends up as large as the data justify. It also has its own boosting (`trials`)
   and can turn the tree into readable rules.

The Python version we use is **`c50py`** (`pip install c50py`, code at `github.com/daviddiazsolis/c50py`),
an open implementation of the algorithm with the scikit-learn interface (`fit`, `predict_proba`, and `plot_tree` to
draw the tree with the same function and format as scikit-learn, but with its own colour palette;
it needs version 0.5.0 or later).
The notebook *Trees 06, C5.0* on the Tree & Ensemble Foundations site goes through it in depth; here we put it at the same
table as the other models. One detail that matters for reading the result: C5.0 prunes by looking at **accuracy**, not
AUC, so it tends to leave few leaves with coarse probabilities.


In [ ]:
import subprocess, sys
try:
    import c50py
    _version_ok = tuple(int(v) for v in c50py.__version__.split(".")[:2]) >= (0, 5)
except ImportError:
    _version_ok = False
if not _version_ok:   # c50py 0.5.0 or later: C5.0's pruning, boosting, winnowing and rulesets
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "c50py>=0.5.0"], check=True)
    import c50py
from c50py import C5Classifier
print("c50py", c50py.__version__)

CATEGORICAS = ["sexo", "educacion", "estado_civil"]
COLS = list(X.columns)
Xo_tr, Xo_te = X_tr.values.astype(object), X_te.values.astype(object)     # c50py receives the raw (unscaled) columns, with the categorical ones as such

for msl in [100, 20]:
    t0 = time.time()
    c5 = C5Classifier(categorical_features=CATEGORICAS, min_samples_leaf=msl).fit(Xo_tr, y_tr, feature_names=COLS)
    evaluar(f"C5.0 (c50py), minimum {msl} per leaf", c5.predict_proba(Xo_tr)[:, 1], c5.predict_proba(Xo_te)[:, 1], time.time() - t0)
    if msl == 100:
        c5_chico = c5

print()
print("The C5.0 tree with minimum 100 per leaf, as rules:")
c5_chico.print_tree(feature_names=COLS)

# The same drawing as the scikit-learn trees of section 4 (same plot_tree arguments)
fig, ax = plt.subplots(figsize=(24, 9))
c5_chico.plot_tree(feature_names=COLS, class_names=["pays", "does not pay"], filled=True, proportion=True, impurity=False, rounded=True, fontsize=8, ax=ax)
ax.set_title("C5.0 (c50py), minimum 100 customers per leaf")
plt.show()

Three readings. First, the C5.0 tree is **smaller and more readable** than the depth-4 CART: with a minimum of 100 per
leaf four leaves remain, all on the delays, and each leaf reads as a business rule ("two months or more of
delay in September: does not pay"). Second, its **AUC is lower** (0.69 with 100 per leaf and 0.68 with 20, against 0.74
for the depth-4 CART), and not because the algorithm is worse but because it optimizes something else: the
pessimistic pruning keeps the tree with the best **accuracy**, and few leaves mean few distinct probabilities,
that is, many ties in the ordering that the AUC measures. It is the difference between a model for explaining and a model
for ranking. Third, the training versus testing gap is among the smallest in the table: pessimistic pruning keeps its
promise not to memorize. On the trees site you can see the opposite case, with categorical variables of many
categories, where C5.0 beats CART with dummies. Further down we see it again with its boosting.


## 5. Ensembles: many trees are worth more than one

An **ensemble** combines the predictions of many models. The intuition is that of asking twenty appraisers for the
price of a house: each has their quirks, but if the quirks are different they cancel out when averaging, and the
average is usually better than almost any of them. There are two ways to build the set, and they are different
ideas.

### Random forest: independent trees, averaged

Hundreds of **deep** trees are trained, each on a different training sample drawn at random
with replacement (so each tree sees a slightly different training set) and, at each question, looking only at a
random subset of the variables (so the trees do not resemble each other). The prediction is the average of
the leaves. Each tree on its own memorizes; the average of many trees that memorized different things does not.
It is the hardest model to ruin there is: there is almost nothing to tune, and adding trees never makes it worse.

### Gradient boosting: small trees, in sequence, correcting

Here the trees are not independent. It starts with a trivial prediction (the base rate), looks at which
customers it got wrong and by how much (the **residuals**), and trains a **small** tree (depth 3, a
stump) to predict those residuals. That tree is added to the prediction, with a small factor called the
**learning rate**, and the process repeats: hundreds of times, each tree correcting what the previous ones left behind.
It is the model that wins almost every competition with tabular data (XGBoost and LightGBM are fast versions
of this idea). Unlike the random forest, it can overfit if given too many rounds, so the
number of trees and the rate are chosen with validation.

Both are trained on the 23 original columns: trees need neither polynomials (they find the thresholds and
interactions themselves) nor scaling.


In [ ]:

t0 = time.time()
bosque = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=0).fit(A_tr, y_tr)
evaluar("random forest, 300 trees", bosque.predict_proba(A_tr)[:, 1], bosque.predict_proba(A_te)[:, 1], time.time() - t0)

t0 = time.time()
gb = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.8, random_state=0).fit(A_tr, y_tr)
evaluar("gradient boosting, 300 trees", gb.predict_proba(A_tr)[:, 1], gb.predict_proba(A_te)[:, 1], time.time() - t0)


In [ ]:
# What they look like inside: one tree of the forest (deep; shown truncated) and the first tree of the boosting (small, on residuals)
un_arbol = bosque.estimators_[0]
print("First tree of the random forest: depth", un_arbol.get_depth(), ", leaves", un_arbol.get_n_leaves(), " (drawn down to depth 3)")
fig, ax = plt.subplots(figsize=(24, 8))
plot_tree(un_arbol, max_depth=3, feature_names=list(X.columns), class_names=["pays", "does not pay"], filled=True, proportion=True, impurity=False, rounded=True, fontsize=8, ax=ax)
ax.set_title("Random forest: the first three levels of tree 1 of 300 (cuts in scaled units; each '(...)' keeps growing)")
plt.show()

primer_gb = gb.estimators_[0, 0]
print("First tree of the gradient boosting: depth", primer_gb.get_depth(), ", leaves", primer_gb.get_n_leaves())
fig, ax = plt.subplots(figsize=(24, 7))
plot_tree(primer_gb, feature_names=list(X.columns), filled=True, impurity=False, rounded=True, fontsize=8, ax=ax)
ax.set_title("Gradient boosting: tree 1 of 300. It predicts residuals (value > 0 pushes toward 'does not pay'), and is added with rate 0.05")
plt.show()

The two drawings show the fundamental difference between the ensembles. The forest tree is **deep** (hundreds of
leaves; only three levels are shown here) and its cuts change from one tree to the next because each one saw a different sample
and a different subset of variables: on its own it memorizes, and the average of 300 that memorized different things is
what generalizes. The boosting tree is **small** (depth 3, eight leaves) and does not predict "pays" or "does not pay": it predicts
a **residual**, how much the accumulated prediction is missing in each group of customers, and its contribution enters multiplied by
the learning rate 0.05. Notice that the first boosting tree starts with `atraso_sep` at the same cut as
the simple tree: the first large residual is precisely the step at two months of delay.

C5.0 also has boosting, older than gradient boosting: each round (`trials`) retrains the tree giving more weight to
the customers the previous rounds misclassified (the AdaBoost idea, with a gentler rule of its own), and the rounds vote. We add it to the
comparison with ten rounds:


In [ ]:
t0 = time.time()
c5_boost = C5Classifier(categorical_features=CATEGORICAS, min_samples_leaf=100, trials=10, random_state=0).fit(Xo_tr, y_tr, feature_names=COLS)
evaluar("C5.0 with boosting, 10 rounds", c5_boost.predict_proba(Xo_tr)[:, 1], c5_boost.predict_proba(Xo_te)[:, 1], time.time() - t0)

With ten rounds the C5.0 goes from 0.69 to 0.77 of AUC in testing: the boosting turns a four-leaf tree into a
model that ranks as well as the random forest (0.773) and stays within a hundredth or less of the gradient boosting, at the price of losing the reading as rules and of a larger training versus testing gap. The table that follows
puts them all together.


In [ ]:

tabla = pd.DataFrame({k: {kk: vv for kk, vv in v.items() if kk != "p_te"} for k, v in RESULTADOS.items()}).T
tabla["AUC 95% interval"] = tabla.apply(lambda r: f"[{r.ic_lo:.3f}, {r.ic_hi:.3f}]", axis=1)
print(tabla[["auc_train", "auc_test", "AUC 95% interval", "recall_1", "precision_1", "captura_20", "seg"]].round(3).to_string())


## 6. Comparing without fooling yourself

The table ranks the models by testing AUC, but before declaring a winner two questions have to be
answered: is the difference larger than the noise? and does it hold up in other samples?

**The bootstrap intervals** are already in the table, and with 9,000 customers they are narrow, on the order of plus or
minus 0.013. But comparing two intervals that overlap a little is not the right test, because the two
models are evaluated on the **same** customers and their errors are correlated. The right test is the
**paired difference**: in each bootstrap resample the AUC of the two models is computed on the same
customers and subtracted; if the interval of the difference does not contain zero, the difference is real.

**Cross-validation** answers the second question: train and measure five times on different folds
of training. It takes a couple of minutes with the boosting.


In [ ]:

def diferencia_pareada(a, b):
    pa, pb = RESULTADOS[a]["p_te"], RESULTADOS[b]["p_te"]
    d = np.array([roc_auc_score(y_te[i], pa[i]) - roc_auc_score(y_te[i], pb[i]) for i in IDX_BS])
    print(f"AUC({a}) - AUC({b}) = {RESULTADOS[a]['auc_test'] - RESULTADOS[b]['auc_test']:+.4f}   paired 95% CI [{np.percentile(d, 2.5):+.4f}, {np.percentile(d, 97.5):+.4f}]")

diferencia_pareada("gradient boosting, 300 trees", "linear logistic")
diferencia_pareada("gradient boosting, 300 trees", "logistic degree 2, L2 by cross-validation")
diferencia_pareada("gradient boosting, 300 trees", "random forest, 300 trees")
diferencia_pareada("logistic degree 2, L2 by cross-validation", "linear logistic")


In [ ]:

modelos_cv = {
    "linear logistic": LogisticRegression(penalty=None, max_iter=5000),
    "random forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=0),
    "gradient boosting": GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.8, random_state=0),
}
print("AUC by cross-validation (5 folds on training):")
for nombre, m in modelos_cv.items():
    a = cross_val_score(m, A_tr, y_tr, cv=cv5, scoring="roc_auc", n_jobs=-1)
    print(f"   {nombre:20s} {a.mean():.3f} +/- {a.std():.3f}   from {a.min():.3f} to {a.max():.3f}")


With this the comparison is settled: the paired interval between boosting and linear logistic does not touch zero, and
the cross-validation ranges of the two do not even come close. The difference is real. Between boosting and random forest
it is smaller, and you have to read the paired interval to decide whether it is distinguishable.


### When the AUC ties, the curve decides

The AUC is an average over the whole ROC curve, and the whole curve includes zones where nobody operates: a bank
does not approve cards while flagging 80% of its good customers. What matters when comparing models is the recall
each one gives **at the false positive rate the business tolerates**: a conservative issuer stops at 5%; a preventive
collections campaign, where flagging too many costs a phone call, can go to 30% or 40%. Two models with almost the
same AUC can cross, one better on the left and the other on the right, and in that case the right question is not
which one has more AUC but which one catches more defaulters at *my* point.

Below, the curves of the seven models zoomed to the operating zone (false positives up to 30%), and a table with
each model's recall at several tolerances plus the **partial AUC** (`roc_auc_score` with `max_fpr`), which summarizes
the curve only up to that rate. Look at the depth-4 tree against the degree-2 logistic: by AUC the tree loses, but at
5% false positives it catches more, because its big step falls right there; at 50% it flips.


In [ ]:

tolerancias = [0.05, 0.10, 0.20, 0.30, 0.50]
filas = []
fig = go.Figure()
for nombre, r in RESULTADOS.items():
    fpr, tpr, _ = roc_curve(y_te, r["p_te"])
    fig.add_scatter(x=fpr, y=tpr, mode="lines", name=f"{nombre} (AUC {r['auc_test']:.3f})")
    fila = {"modelo": nombre, "AUC": r["auc_test"]}
    for f in tolerancias:
        fila[f"recall at FP {int(f*100)}%"] = float(np.interp(f, fpr, tpr))
        fila[f"partial AUC up to {int(f*100)}%"] = roc_auc_score(y_te, r["p_te"], max_fpr=f)
    filas.append(fila)
fig.add_scatter(x=[0, 0.3], y=[0, 0.3], mode="lines", name="chance", line=dict(dash="dash", color="gray"))
fig.update_layout(title="ROC curves on testing, zoomed to the operating zone (FP up to 30%)", xaxis_title="false positive rate",
                  yaxis_title="recall", xaxis_range=[0, 0.3], yaxis_range=[0, 0.8], height=520, width=760)
fig.show()
tabla_roc = pd.DataFrame(filas).set_index("modelo")
print("Recall at each false positive tolerance (who catches more defaulters at each point):")
print(tabla_roc[[c for c in tabla_roc.columns if c.startswith("recall")] + ["AUC"]].round(3).sort_values("recall at FP 5%", ascending=False).to_string())
print()
print("Partial AUC (scikit-learn, max_fpr; standardized so that 0.5 is chance and 1 is perfect):")
print(tabla_roc[[c for c in tabla_roc.columns if c.startswith("partial AUC")]].round(3).to_string())


Reading: at 5% false positives the depth-4 tree ties with the boosting and beats the random forest and the degree-2
logistic, even though by AUC it is fourth; at 50% it is last in that group. Random forest and boosting, 0.007 apart in
AUC, are indistinguishable at 5% and only separate further to the right. The AUC ranks on average; your operating point
may rank differently, and that is why it pays to look at the curve (or the partial AUC) in the zone where you will
actually work.


## 7. What the boosting found that the logistic could not see

This is the part that is worth understanding well. Let's take the most important variable, `atraso_sep`, and compare,
for each value it takes, three things: what percentage of those customers **actually** did not pay, what average risk
the linear logistic assigned them, and what the boosting assigned them.


In [ ]:

comp = pd.DataFrame({"atraso_sep": X_te["atraso_sep"].values, "realmente_no_pago": y_te,
                     "logistica": RESULTADOS["linear logistic"]["p_te"], "grado_2": RESULTADOS["logistic degree 2, L2 by cross-validation"]["p_te"],
                     "boosting": RESULTADOS["gradient boosting, 300 trees"]["p_te"]})
por_atraso = comp.groupby("atraso_sep").agg(clientes=("realmente_no_pago", "size"), realmente_no_pago=("realmente_no_pago", "mean"),
                                            logistica=("logistica", "mean"), grado_2=("grado_2", "mean"), boosting=("boosting", "mean"))
print(por_atraso[por_atraso["clientes"] >= 20].round(3))


Read the `realmente_no_pago` column from top to bottom. Among the customers who are up to date (delay 0 or negative) the
default rate hovers around 13 to 17%, almost flat. With **one** month late it jumps to about 33%. With **two** months it jumps to
about 70%, and from there on it stays there. It is a **staircase with two steps**, not a ramp.

The linear logistic cannot draw a staircase: since $z$ is linear in `atraso_sep`, it is forced to raise the
risk at the same pace for each month of delay, and it gets both ends wrong: it underestimates the risk of those who
are up to date with a negative delay, overestimates that of those with delay 0, and falls short on the step at
2. The degree-2 one improves because a square can bend the ramp, but it still cannot make a step. The
boosting reproduces each step almost to the cent, because a tree does exactly that: it asks "delay greater than or
equal to 2?" and assigns a different risk to each side. Let's see it as a curve, taking a typical customer and varying
only the September delay.


In [ ]:

tipico = X_tr.median()
grilla = pd.DataFrame([tipico] * 11); grilla["atraso_sep"] = np.arange(-2, 9); grilla = grilla[X.columns]
G = esc.transform(grilla)
curva = pd.DataFrame({"atraso_sep": grilla["atraso_sep"],
                      "linear logistic": logit.predict_proba(G)[:, 1],
                      "logistic degree 2": poli_cv.predict_proba(esc2.transform(poly.transform(G)))[:, 1],
                      "gradient boosting": gb.predict_proba(G)[:, 1]}).melt(id_vars="atraso_sep", var_name="modelo", value_name="riesgo estimado")
fig = px.line(curva, x="atraso_sep", y="riesgo estimado", color="modelo", markers=True, title="Estimated risk by months of delay, for a typical customer")
fig.update_layout(height=400); fig.show()


### Which variables each one uses

The logistic has coefficients; the boosting reports the **importance** of each variable, which is how much
each one contributed to reducing impurity across all the trees (a fraction that sums to 1). They are not the
same thing and cannot be compared in magnitude, but they can be compared in **order**: if two models from different families
agree on which variables come first, the list can be taken to a meeting.


In [ ]:

imp = pd.DataFrame({"logistic coef. (abs)": np.abs(logit.coef_[0]), "boosting importance": gb.feature_importances_, "random forest importance": bosque.feature_importances_}, index=X.columns)
imp["logistic ranking"] = imp["logistic coef. (abs)"].rank(ascending=False).astype(int)
imp["boosting ranking"] = imp["boosting importance"].rank(ascending=False).astype(int)
print(imp.sort_values("boosting importance", ascending=False).head(10).round(3))


## 8. Using the winner: cost-based threshold and calibration

A higher AUC is not a decision. To use the model you have to set the threshold, and from the previous notebook
we know how: with the costs of each error. Suppose letting through a customer who does not pay costs 4 times
what rejecting one who would have paid costs ($c_{FN} = 4$, $c_{FP} = 1$). The formula says threshold $= 1/5 = 0.2$ if
the probabilities are calibrated; the table on testing says where the minimum really falls. And since here
we have 9,000 testing customers, the two should agree more closely than in German Credit.

We also check calibration by deciles: tree ensembles do not always deliver well calibrated
probabilities (the random forest, in particular, tends never to get close to 0 or 1 because it averages), and if the
threshold is chosen by formula that matters.


In [ ]:

c_FN, c_FP = 4, 1
umbrales = np.abs(np.round(np.arange(1.0, -0.001, -0.05), 2))
def costo_por_umbral(p):
    filas = []
    for u in umbrales:
        pred = (p > u).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_te, pred, labels=[0, 1]).ravel()
        filas.append(dict(umbral=u, costo=fp * c_FP + fn * c_FN, recall=tp / (tp + fn), precision=tp / (tp + fp) if tp + fp else 0))
    return pd.DataFrame(filas)

print("Threshold by formula:", round(c_FP / (c_FP + c_FN), 3))
for nombre in ["linear logistic", "logistic degree 2, L2 by cross-validation", "random forest, 300 trees", "gradient boosting, 300 trees"]:
    t = costo_por_umbral(RESULTADOS[nombre]["p_te"]); f = t.loc[t.costo.idxmin()]
    print(f"{nombre:45s} optimal threshold (table) {f.umbral:.2f}   minimum cost {int(f.costo)}   recall {f.recall:.3f}  precision {f.precision:.3f}   (cost with threshold 0.5: {int(t.loc[t.umbral == 0.5, 'costo'].iloc[0])})")


In [ ]:

cal = pd.DataFrame({"y": y_te, "logistic": RESULTADOS["linear logistic"]["p_te"], "random forest": RESULTADOS["random forest, 300 trees"]["p_te"], "boosting": RESULTADOS["gradient boosting, 300 trees"]["p_te"]})
out = {}
for m in ("logistic", "random forest", "boosting"):
    g = pd.qcut(cal[m], 10, labels=False, duplicates="drop")
    out[m + ": mean p"] = cal.groupby(g)[m].mean().values
    out[m + ": actual"] = cal.groupby(g)["y"].mean().values
print("Calibration by risk deciles (mean p of the decile against actual fraction that defaulted):")
print(pd.DataFrame(out).round(3).to_string())


Reading the calibration: the boosting and the random forest are well calibrated in every decile (here the
random forest did not have the problem of shrinking toward the center, because with 21,000 customers and leaves of 5 its
trees go far). The worst calibrated is the linear logistic: in the middle deciles it assigns between 0.18 and 0.23
where reality is 0.12 to 0.15, and in the top decile it assigns 0.59 where reality is 0.67. It is the same
staircase from section 7 seen from another side: the line spreads the risk along a ramp and reality
concentrates it on a step. That is why its table threshold (0.25) moves away from the formula (0.20) and the boosting's does not.

The model to be used is the boosting, with the threshold that came out of the cost table, and with the calibration
checked. In business terms that translates into how many customers it rejects and how many defaults it avoids, which is
what the recall and precision row at that threshold says. An exercise below asks to measure how much is saved relative
to the logistic in cost units.


## 9. What to take away

1. Regularization matters when the model has many weights **relative to the rows**: with 299 degree-2 columns
   and 21,000 customers it changes almost nothing; with the same 299 columns and 1,000 customers it is what separates
   a model that generalizes from one that memorizes. In German Credit, with 18 columns and 700 rows, there was nothing to
   rein in.
2. A **tree** is a sequence of questions; it draws steps and memorizes if allowed to grow. A single pruned tree
   does not beat a good logistic.
3. An **ensemble** turns many trees into a good model: random forest averages independent, deep trees
   (almost impossible to ruin); gradient boosting adds small trees that correct each other in sequence
   (the most accurate with tabular data, and the only one of the two that can overfit with too many rounds).
4. **Comparing** requires intervals: bootstrap for each model, paired difference between two, cross-validation
   to see whether it holds up. With 9,000 testing customers, hundredths of AUC are real; with 300, they are not.
5. When a model wins, you have to **look at what it found**. Here it was a staircase in the months of delay, and that
   shape explains why a line loses, a polynomial half draws, and a tree gets it right.
6. An AUC is not a decision: the model is used with a **threshold chosen with costs**, and with the calibration
   checked, because ensembles do not always deliver honest probabilities.


## Exercises

**Exercise 1.** Train the gradient boosting with 30, 100, 300 and 1,000 trees (same rate 0.05). Report training
and testing AUC for each. Does it always improve? Where does the gap start to open? Compare with the random
forest at the same four sizes: does the same thing happen to it?

**Exercise 2.** Repeat the table of section 7 for `edad` (age; group it into 5-year brackets). Is there a nonlinear
shape there too, or is it a ramp the logistic already draws well?

**Exercise 3.** With $c_{FN} = 4$ and $c_{FP} = 1$, compute the total cost in testing of the boosting and of the
linear logistic, each with its optimal threshold from the table. How many cost units are saved with the
boosting over 9,000 customers? Put a paired bootstrap interval on that saving.

**Exercise 4.** Train the boosting with only the first 1,000 training customers. Measure training and
testing AUC. Does it still beat the linear logistic trained on the same 1,000? Before running it, write down
what you expect, and then compare with what happened in German Credit, where with 1,000 customers nothing beat the
straight line.


## Solutions


In [ ]:

# SOLUTION 1: number of trees
print("gradient boosting (rate 0.05, depth 3):")
for n in [30, 100, 300, 1000]:
    m = GradientBoostingClassifier(n_estimators=n, learning_rate=0.05, max_depth=3, subsample=0.8, random_state=0).fit(A_tr, y_tr)
    print(f"   {n:5d} trees  AUC train {roc_auc_score(y_tr, m.predict_proba(A_tr)[:, 1]):.3f}  test {roc_auc_score(y_te, m.predict_proba(A_te)[:, 1]):.3f}")
print("random forest:")
for n in [30, 100, 300, 1000]:
    m = RandomForestClassifier(n_estimators=n, min_samples_leaf=5, n_jobs=-1, random_state=0).fit(A_tr, y_tr)
    print(f"   {n:5d} trees  AUC train {roc_auc_score(y_tr, m.predict_proba(A_tr)[:, 1]):.3f}  test {roc_auc_score(y_te, m.predict_proba(A_te)[:, 1]):.3f}")
print("Reading: in the boosting the testing AUC rises and then stalls or falls while the training one keeps rising (the gap opens); in the random forest adding trees only stabilizes.")


In [ ]:

# SOLUTION 2: age
comp["tramo_edad"] = (X_te["edad"].values // 5) * 5
por_edad = comp.groupby("tramo_edad").agg(clientes=("realmente_no_pago", "size"), realmente_no_pago=("realmente_no_pago", "mean"), logistica=("logistica", "mean"), boosting=("boosting", "mean"))
print(por_edad[por_edad.clientes >= 50].round(3))
print("Reading: age is almost flat or a gentle ramp; the logistic and the boosting assign it almost the same. The nonlinearity that matters is in the delays, not here.")


In [ ]:

# SOLUTION 3: cost saving with each model's optimal threshold
def costo_min(p, idx=None):
    yy = y_te if idx is None else y_te[idx]; pp = p if idx is None else p[idx]
    mejor = None
    for u in umbrales:
        pred = (pp > u).astype(int); tn, fp, fn, tp = confusion_matrix(yy, pred, labels=[0, 1]).ravel()
        c = fp * c_FP + fn * c_FN
        if mejor is None or c < mejor: mejor = c
    return mejor
pb, pl = RESULTADOS["gradient boosting, 300 trees"]["p_te"], RESULTADOS["linear logistic"]["p_te"]
ahorro = costo_min(pl) - costo_min(pb)
d = np.array([costo_min(pl, i) - costo_min(pb, i) for i in IDX_BS[:200]])
print(f"Minimum cost logistic: {costo_min(pl)}   boosting: {costo_min(pb)}   saving: {ahorro} units over 9,000 customers   paired 95% CI [{np.percentile(d, 2.5):.0f}, {np.percentile(d, 97.5):.0f}]")


In [ ]:

# SOLUTION 4: with only 1,000 training customers
A_chico, y_chico = A_tr[:1000], y_tr[:1000]
gb_chico = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, subsample=0.8, random_state=0).fit(A_chico, y_chico)
lg_chico = LogisticRegression(penalty=None, max_iter=5000).fit(A_chico, y_chico)
for nombre, m in [("boosting with 1,000", gb_chico), ("logistic with 1,000", lg_chico)]:
    print(f"{nombre:22s} AUC train {roc_auc_score(y_chico, m.predict_proba(A_chico)[:, 1]):.3f}  test {roc_auc_score(y_te, m.predict_proba(A_te)[:, 1]):.3f}")
print("Reading: with 1,000 customers the boosting memorizes (training AUC close to 1) and its testing AUC falls from 0.78 to 0.75, but it still beats the logistic by about 0.05. The difference with German Credit is not just size: here there is a strong, simple staircase in the delays that even a boosting with little data finds, and there there was nothing nonlinear to find. The ensemble wins when there is something to discover; what abundant data does is close the gap between training and testing.")
